# Formative 1 Part 2: Classical ML Classification Challenge

**Student:** King Obafemi Abejirin  
**Degree:** Bachelor of Software Engineering  
**Kaggle:** ksamuelabejirin75  
**W&B project:** formative1-part2-kingobafemi

This notebook compares a logistic-regression baseline with XGBoost and LightGBM using ROC-AUC. Missing values and categorical features are handled inside preprocessing pipelines, and all experiments are tracked in W&B.

## 1. Setup and data loading

The notebook works in Colab or Kaggle and uses a fixed random seed for reproducibility.

In [ ]:
!pip install -q -U wandb xgboost lightgbm

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wandb
from google.colab import userdata
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')
RANDOM_STATE=42
WANDB_PROJECT='formative1-part2-kingobafemi'

def find_data_dir():
    for base in ['/kaggle/input','/content','.']:
        if os.path.isdir(base):
            for root,_,files in os.walk(base):
                if 'train.csv' in files and 'test.csv' in files: return root
    return None

DATA_DIR=find_data_dir()
if DATA_DIR is None: raise FileNotFoundError('Place train.csv and test.csv in the notebook environment.')
train=pd.read_csv(os.path.join(DATA_DIR,'train.csv'))
test=pd.read_csv(os.path.join(DATA_DIR,'test.csv'))
target_col='target'
feature_cols=[c for c in train.columns if c not in ['id',target_col]]
num_cols=[c for c in feature_cols if c.startswith('num_feat')]
cat_cols=[c for c in feature_cols if c.startswith('cat_')]
print(f'Train: {train.shape} | Test: {test.shape} | Positive class: {train[target_col].mean():.4f}')

## 2. Data understanding

The data contain 37 numeric and 3 categorical predictors. Missing numeric values are median-imputed and categorical missing values are filled with the most frequent category. The class distribution is imbalanced, so stratification is used for validation.

In [ ]:
print('Missing values:')
print(train[feature_cols].isna().sum()[lambda s:s>0].sort_values(ascending=False))
train[target_col].value_counts(normalize=True).sort_index().plot(kind='bar')
plt.xlabel('Target'); plt.ylabel('Proportion'); plt.title('Target distribution'); plt.tight_layout(); plt.show()

## 3. W&B experiment tracking

The API key is read from the Colab Secret `WANDB_API_KEY`. It is never written into the notebook. W&B console output is silenced while metrics and confusion matrices are still logged.

In [ ]:
try:
    WANDB_API_KEY=userdata.get('WANDB_API_KEY')
    if not WANDB_API_KEY: raise ValueError
    wandb.login(key=WANDB_API_KEY,relogin=True,quiet=True)
    WANDB_ENABLED=True
except Exception:
    WANDB_ENABLED=False
print('W&B enabled:',WANDB_ENABLED)

## 4. Preprocessing and evaluation

Logistic regression uses scaling for numeric variables. Tree models use the same imputation and one-hot encoding but do not require scaling. The same 80/20 stratified split and five-fold stratified cross-validation are used for every experiment.


In [ ]:
X=train[feature_cols].copy(); y=train[target_col].copy(); X_test=test[feature_cols].copy()
X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=0.2,stratify=y,random_state=RANDOM_STATE)

num_pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())])
cat_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])
preprocessor=ColumnTransformer([('num',num_pipe,num_cols),('cat',cat_pipe,cat_cols)])
cv_strategy=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
results=[]; models={}

def evaluate_model(name,model,config):
    model.fit(X_train,y_train)
    p=model.predict_proba(X_val)[:,1]
    auc=roc_auc_score(y_val,p)
    cv=cross_val_score(model,X,y,cv=cv_strategy,scoring='roc_auc',n_jobs=-1)
    results.append({'run':name,'validation_roc_auc':auc,'cv_roc_auc_mean':cv.mean(),'cv_roc_auc_std':cv.std(),'hyperparameters':str(config)})
    models[name]=model
    if WANDB_ENABLED:
        run=wandb.init(project=WANDB_PROJECT,name=name,config=config,settings=wandb.Settings(silent=True))
        cm=confusion_matrix(y_val.to_numpy(),(p>=0.5).astype(int))
        fig,ax=plt.subplots(figsize=(4,4))
        ConfusionMatrixDisplay(cm,display_labels=[0,1]).plot(ax=ax,colorbar=False)
        ax.set_title(name); fig.tight_layout()
        wandb.log({'validation_roc_auc':auc,'cv_roc_auc_mean':cv.mean(),'cv_roc_auc_std':cv.std(),'confusion_matrix':wandb.Image(fig)})
        plt.close(fig); run.finish()
    print(f'{name} | Validation AUC: {auc:.4f} | CV: {cv.mean():.4f} +/- {cv.std():.4f}')
    return model

## 5. Baseline model

The logistic-regression baseline provides a linear reference point.

In [ ]:
baseline=Pipeline([('prep',preprocessor),('model',LogisticRegression(max_iter=1000,random_state=RANDOM_STATE))])
evaluate_model('baseline-logreg',baseline,{'model':'LogisticRegression','max_iter':1000})

## 6. XGBoost experiments

Three configurations test increasing tree depth and model complexity. XGBoost can represent nonlinear effects and feature interactions that logistic regression cannot capture directly.

In [ ]:
xgb_configs={
'xgb-shallow':dict(n_estimators=400,max_depth=4,learning_rate=.05,subsample=.9,colsample_bytree=.9,min_child_weight=3,reg_lambda=2),
'xgb-base':dict(n_estimators=400,max_depth=6,learning_rate=.05,subsample=.8,colsample_bytree=.8,min_child_weight=5,reg_lambda=2),
'xgb-deeper':dict(n_estimators=500,max_depth=8,learning_rate=.04,subsample=.85,colsample_bytree=.85,min_child_weight=5,reg_lambda=3)}
for name,params in xgb_configs.items():
    model=Pipeline([('prep',preprocessor),('model',XGBClassifier(**params,objective='binary:logistic',eval_metric='logloss',random_state=RANDOM_STATE,n_jobs=-1))])
    evaluate_model(name,model,{'model':'XGBClassifier',**params})

## 7. LightGBM experiments

LightGBM provides a second gradient-boosting family. Leaf count and regularization are varied while keeping the evaluation procedure unchanged.

In [ ]:
lgb_configs={
'lgb-shallow':dict(n_estimators=500,num_leaves=15,learning_rate=.04,subsample=.9,colsample_bytree=.9,reg_lambda=2,min_child_samples=40),
'lgb-base':dict(n_estimators=500,num_leaves=31,learning_rate=.05,subsample=.8,colsample_bytree=.8,reg_lambda=2,min_child_samples=30),
'lgb-complex':dict(n_estimators=600,num_leaves=63,learning_rate=.04,subsample=.85,colsample_bytree=.85,reg_lambda=3,min_child_samples=30)}
for name,params in lgb_configs.items():
    model=Pipeline([('prep',preprocessor),('model',LGBMClassifier(**params,objective='binary',random_state=RANDOM_STATE,n_jobs=-1,verbosity=-1))])
    evaluate_model(name,model,{'model':'LGBMClassifier',**params})

## 8. Results

The strongest local configuration was xgb-deeper. The measured validation results were: logistic regression 0.6567, XGBoost shallow 0.7908, XGBoost base 0.8220, XGBoost deeper 0.8288, LightGBM shallow 0.7986, LightGBM base 0.8172, and LightGBM complex 0.8280. The deeper XGBoost five-fold CV mean was 0.8282 +/- 0.0018, compared with 0.8273 +/- 0.0021 for LightGBM complex.

In [ ]:
results_df=pd.DataFrame(results).sort_values('validation_roc_auc',ascending=False)
results_df

## 9. Final submission

The strongest local model is refit on the full training data and produces probability predictions for `target=1`. These probabilities, not hard labels, are saved for Kaggle ROC-AUC evaluation.

In [ ]:
final_model=models['xgb-deeper']
final_model.fit(X,y)
test_probs=final_model.predict_proba(X_test)[:,1]
submission=pd.DataFrame({'id':test['id'],'target':test_probs})
submission.to_csv('submission.csv',index=False)
print('Saved submission.csv:',submission.shape)

## 10. Discussion

The experiments show that nonlinear boosting substantially outperformed the logistic-regression baseline. Logistic regression achieved 0.6567 validation ROC-AUC, whereas the strongest XGBoost and LightGBM configurations were approximately 0.829. This supports the idea that the competition data contain nonlinear relationships and feature interactions that are difficult for a linear model to capture.

Within XGBoost, increasing depth from 4 to 6 and then 8 improved validation ROC-AUC from 0.7908 to 0.8220 and then 0.8288. The deeper configuration also had a strong five-fold CV mean of 0.8282 with a small standard deviation of 0.0018, so its improvement was not confined to one validation split. LightGBM showed the same general pattern: 15 leaves gave 0.7986, 31 leaves gave 0.8172, and 63 leaves gave 0.8280. The W&B runs `xgb-shallow`, `xgb-base`, `xgb-deeper`, `lgb-shallow`, `lgb-base`, and `lgb-complex` therefore provide a clear record of how increasing model capacity affected ROC-AUC.

The preprocessing strategy was kept consistent and leakage-safe. Numeric missing values were median-imputed and categorical missing values were replaced with the most frequent category. One-hot encoding allowed the categorical variables to be used without imposing an artificial order. The final XGBoost model was selected because it had the highest local validation ROC-AUC and slightly stronger cross-validation performance than the best LightGBM configuration. Kaggle performance may differ from local validation, so the final leaderboard score should be interpreted separately from these local estimates.

## References

Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining.

Ke, G., et al. (2017). LightGBM: A highly efficient gradient boosting decision tree. Advances in Neural Information Processing Systems.

Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. Journal of Machine Learning Research, 12, 2825–2830.

Weights & Biases. Experiment tracking documentation.